In [ ]:
import os, re, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FormatStrFormatter  # <-- NEW
from matplotlib.ticker import MaxNLocator, ScalarFormatter, FuncFormatter  # NEW


In [ ]:

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman No9 L", "DejaVu Serif"],  # fallbacks
    "mathtext.fontset": "dejavuserif",   # math text harmonizes with serif
    "axes.unicode_minus": False,         # proper minus sign with serif fonts
    "pdf.fonttype": 42,                  # embed TrueType in vector outputs
    "ps.fonttype": 42,
})

# ---------------- Config ----------------
PER_IMAGE_DIRS = [
    os.path.join("final_result", "per_model_metrics"),
    "per_model_metrics",
]
AGG_DIR = "final_result"  # Aggregated_Summary_Core{i}.csv live here

OUT_DIR = os.path.join("final_result", "per_image_histograms_uniform")
os.makedirs(OUT_DIR, exist_ok=True)

# Two requested outputs
OUT_PNG_SET1 = os.path.join(OUT_DIR, "HIST_7x4_MAE_GMSD_LPIPS_DISTS.png")
OUT_PNG_SET2 = os.path.join(OUT_DIR, "HIST_7x7_R2_SoftDice_PSNR_VIF_SSIM_MSSSIM_FSIM.png")

GROUPS = list(range(1, 8))  # 1..7 cores
GROUP_LABEL = lambda r: f"{r} core" if r == 1 else f"{r} cores"

# Column order & mapping (DisplayName -> per_image column name)
METRICS_SET1 = [
    ("MAE",       "mae"),
    ("GMSD",      "gmsd"),
    ("LPIPS",     "lpips"),
    ("DISTS",     "dists"),
]
METRICS_SET2 = [
    ("R2",        "r2"),
    ("Soft Dice", "soft_dice"),
    ("PSNR",      "psnr"),
    ("VIF",       "vif"),
    ("SSIM",      "ssim"),
    ("MS-SSIM",   "ms_ssim"),
    ("FSIM",      "fsim"),
]

# Put stats box on the RIGHT for these (per-image column names, lowercase)
RIGHT_ANNOT_METRICS = {"mae", "gmsd", "lpips", "dists"}

# -------- Per-metric x-limits (yours) --------
XLIMITS = {
    "mae":      (0.02, 0.08),
    "gmsd":     (0.1, 0.19),
    "lpips":    (0.19, 0.31),
    "dists":    (0.1, 0.22),
    "r2":       (0.85, 1.0),
    "soft_dice":(0.985, 1.0),
    "psnr":     (17.5, 27.5),
    "vif":      (0.22, 0.33),
    "ssim":     (0.75, 0.88),
    "ms_ssim":  (0.64, 0.88),
    "fsim":     (0.85, 0.92),
}

# -------- NEW: Per-metric y-limits (counts). Add only what you want to fix; others auto. --------
# Example: {"mae": (0, 300), "psnr": (0, 120)}

YLIMITS_ROW = {
     1: (0, 10000),
     2: (0, 30000),
     3: (0, 50000),
     4: (0, 50000),
     5: (0, 30000),
     6: (0, 10000),
     7: (0, 1500),
}

# -------- Scientific notation for annotation text ONLY --------
SCI_NOTATION_METRICS = {"mae", "gmsd", "lpips", "dists"}  # annotation text only
SCI_PREC = 2  # x.xxE±yy
SIGFIGS_X = 2

# Histogram style
BINS_MODE   = "count"  # "count" or "width"
N_BINS      = 20
BIN_WIDTH   = None
PAD_FRAC    = 0.02
DENSITY     = False

# Mean line style (red & thicker)
DRAW_MEAN   = True
MEAN_LINE_KW = dict(color="red", linestyle="--", linewidth=2.0)

# Bar style (outline only)
EDGE_COLOR  = "black"
EDGE_WIDTH  = 0.8

# Figure sizing helpers (scale by grid)
HEIGHT_PER_ROW = 2.2
WIDTH_PER_COL  = 2.5
DPI            = 250
TITLE_FONTSIZE = 14
TEXT_FONTSIZE  = 12
TICK_FONTSIZE  = 12
LABEL_FONTSIZE = 12

# -------- NEW: x-axis tick target (4–5 ticks) --------
X_TICKS_TARGET = 3  # MaxNLocator will aim for this; min_n_ticks=4 enforced below.

# ---------------- Helpers ----------------

def make_sigfig_formatter(sig=2):
    """Return a FuncFormatter that prints numbers with `sig` significant digits,
    using plain decimals (no scientific notation), trimming trailing zeros."""
    def _fmt(x, pos=None):
        if not np.isfinite(x) or x == 0:
            return "0"
        # round to `sig` significant digits
        dec = sig - int(np.floor(np.log10(abs(x)))) - 1
        x_rounded = round(x, dec)
        # fixed-point string with the needed decimals (>=0)
        dec_out = max(0, dec)
        s = f"{x_rounded:.{dec_out}f}"
        # clean up trailing zeros and lone decimal points
        if "." in s:
            s = s.rstrip("0").rstrip(".")
        return s
    return FuncFormatter(_fmt)


def find_per_image_dir():
    for d in PER_IMAGE_DIRS:
        if os.path.isdir(d):
            return d
    raise FileNotFoundError(f"Could not find per_model_metrics dir in: {PER_IMAGE_DIRS}")

def parse_group_from_name(fname):
    m = re.search(r"_G(\d+)_", os.path.basename(fname), flags=re.IGNORECASE)
    return int(m.group(1)) if m else None

def index_files_by_group():
    base = find_per_image_dir()
    files = sorted(glob.glob(os.path.join(base, "*_per_image.csv")))
    mp = {r: [] for r in GROUPS}
    for fp in files:
        r = parse_group_from_name(fp)
        if r in mp:
            mp[r].append(fp)
    return mp

def load_col(fp, col):
    try:
        head = pd.read_csv(fp, nrows=1)
        if col not in head.columns:
            return None
        vals = pd.read_csv(fp, usecols=[col])[col].to_numpy()
        vals = pd.to_numeric(vals, errors="coerce")
        vals = vals[~np.isnan(vals)]
        return vals if vals.size else None
    except Exception:
        return None

def pooled_values(metric_col, files):
    bags = []
    for fp in files:
        v = load_col(fp, metric_col)
        if v is not None and v.size:
            bags.append(v)
    if not bags:
        return np.array([], dtype=float)
    return np.concatenate(bags, axis=0).astype(float)

def _auto_lo_hi(metric_col, files_by_group):
    mins, maxs = [], []
    for r in GROUPS:
        arr = pooled_values(metric_col, files_by_group[r])
        if arr.size:
            mins.append(np.nanmin(arr)); maxs.append(np.nanmax(arr))
    if not mins:
        return None, None
    lo, hi = float(np.min(mins)), float(np.max(maxs))
    if hi <= lo:
        span = max(abs(lo), 1.0) * 1e-3
        lo, hi = lo - span, hi + span
    pad = (hi - lo) * PAD_FRAC
    return lo - pad, hi + pad

def global_range_and_bins(metric_col, files_by_group):
    key = metric_col.lower().strip()
    override = XLIMITS.get(key, None)
    if override is not None:
        lo, hi = override
        if lo is None or hi is None:
            lo, hi = _auto_lo_hi(metric_col, files_by_group)
    else:
        lo, hi = _auto_lo_hi(metric_col, files_by_group)

    if lo is None or hi is None:
        return None, None, None

    if BINS_MODE == "count":
        edges = np.linspace(lo, hi, N_BINS + 1)
    elif BINS_MODE == "width" and BIN_WIDTH and BIN_WIDTH > 0:
        start = np.floor(lo / BIN_WIDTH) * BIN_WIDTH
        end   = np.ceil(hi / BIN_WIDTH)  * BIN_WIDTH
        edges = np.arange(start, end + BIN_WIDTH, BIN_WIDTH)
    else:
        raise ValueError("Set BINS_MODE to 'count' (with N_BINS) or to 'width' (with BIN_WIDTH>0)")
    return lo, hi, edges

def avg_std_across_models(metric_col, group_label):
    vals = []
    for i in range(1, 9):
        p = os.path.join(AGG_DIR, f"Aggregated_Summary_Core{i}.csv")
        if not os.path.exists(p): continue
        try:
            df = pd.read_csv(p)
            df["metric"] = df["metric"].str.lower().str.strip()
            df["group"]  = df["group"].str.lower().str.strip()
            row = df[(df["metric"] == metric_col.lower()) & (df["group"] == group_label.lower())]
            if "mean_std_across_models" in row.columns and not row.empty:
                v = pd.to_numeric(row["mean_std_across_models"], errors="coerce").dropna()
                vals.extend(v.tolist())
        except Exception:
            pass
    return float(np.nanmean(vals)) if len(vals) else float("nan")

def fmt_stat(val, metric_col):
    # Keep scientific for annotation only (as you originally wanted).
    return f"{val:.{SCI_PREC}E}"
    """
    if metric_col.lower() in SCI_NOTATION_METRICS:
        return f"{val:.{SCI_PREC}E}"
    else:
        return f"{val:.3g}"
    """

def draw_hist_grid(metrics_list, out_png, files_by_group):
    n_rows = len(GROUPS)
    n_cols = len(metrics_list)
    figsize = (max(10, WIDTH_PER_COL * n_cols), HEIGHT_PER_ROW * n_rows)

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=figsize, dpi=DPI)
    if n_rows == 1 and n_cols == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    elif n_cols == 1:
        axes = axes.reshape(-1, 1)

    # --- Precompute x-bin edges per metric once (consistent across rows) ---
    edges_by_metric = {}
    lohi_by_metric  = {}
    for (disp_name, col_name) in metrics_list:
        lo, hi, edges = global_range_and_bins(col_name, files_by_group)
        edges_by_metric[col_name] = edges
        lohi_by_metric[col_name]  = (lo, hi)

    # --- Determine per-row y-limits (override or auto from all metrics in that row) ---
    row_ylims = {}
    for r in GROUPS:
        if r in YLIMITS_ROW and YLIMITS_ROW[r] is not None:
            row_ylims[r] = YLIMITS_ROW[r]
            continue

        # Auto: compute max height across *all* metrics in this row
        ymax = 0.0
        for (_, col_name) in metrics_list:
            edges = edges_by_metric[col_name]
            if edges is None: 
                continue
            arr = pooled_values(col_name, files_by_group[r])
            if arr.size:
                counts, _ = np.histogram(arr, bins=edges, density=DENSITY)
                if counts.size:
                    ymax = max(ymax, float(np.nanmax(counts)))
        if ymax > 0:
            row_ylims[r] = (0, ymax * (1 + YLIM_PAD_FRAC))
        else:
            row_ylims[r] = None  # no data in this row

    # --- Plot ---
    for c, (disp_name, col_name) in enumerate(metrics_list):
        lo, hi = lohi_by_metric[col_name]
        edges  = edges_by_metric[col_name]

        if edges is None:
            for r_idx, r in enumerate(GROUPS):
                ax = axes[r_idx, c]; ax.axis("off")
                if r_idx == 0: ax.set_title(disp_name, fontsize=TITLE_FONTSIZE)
            continue

        for r_idx, r in enumerate(GROUPS):
            ax = axes[r_idx, c]
            group_files = files_by_group[r]
            arr = pooled_values(col_name, group_files)

            if arr.size:
                mu = float(np.nanmean(arr))
                sd_models = avg_std_across_models(col_name, GROUP_LABEL(r))

                ax.hist(
                    arr, bins=edges, density=DENSITY,
                    facecolor="none", edgecolor=EDGE_COLOR, linewidth=EDGE_WIDTH
                )
                if DRAW_MEAN:
                    ax.axvline(mu, **MEAN_LINE_KW)

                if col_name.lower() in RIGHT_ANNOT_METRICS:
                    x_anch, ha = 0.97, "right"
                else:
                    x_anch, ha = 0.03, "left"

                ax.text(
                    x_anch, 0.95,
                    f"Mean: {fmt_stat(mu, col_name)}\nSD: {fmt_stat(sd_models, col_name)}",
                    transform=ax.transAxes, va="top", ha=ha, fontsize=TEXT_FONTSIZE,
                    bbox=dict(boxstyle="round", facecolor="white", alpha=0.75, lw=0)
                )
            else:
                ax.text(0.5, 0.5, "No data", ha="center", va="center", fontsize=TEXT_FONTSIZE)

            # Enforce per-metric xlimits
            ax.set_xlim(lo, hi)

            # NEW: enforce per-row ylimits
            if row_ylims[r] is not None:
                ax.set_ylim(*row_ylims[r])

            # X-axis: 4–5 ticks, plain decimals (no scientific)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=X_TICKS_TARGET, min_n_ticks=4))
            fmt = ScalarFormatter(useOffset=False, useMathText=False)
            fmt.set_scientific(False)
            ax.xaxis.set_major_formatter(fmt)
            ax.ticklabel_format(axis='x', style='plain', useOffset=False)

            # tidy ticks & labels
            if r_idx < n_rows - 1:
                ax.set_xticklabels([])
            else:
                ax.tick_params(axis='x', labelsize=TICK_FONTSIZE)
            if c != 0:
                ax.set_yticklabels([])
            else:
                ax.tick_params(axis='y', labelsize=TICK_FONTSIZE)

            if r_idx == 0:
                ax.set_title(disp_name, fontsize=TITLE_FONTSIZE)
            if c == 0:
                ax.set_ylabel(GROUP_LABEL(r), fontsize=LABEL_FONTSIZE)

    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    fig.savefig(out_png)
    plt.close(fig)
    print(f"[SAVE] {out_png}")

# ---------------- Main ----------------
def main():
    files_by_group = index_files_by_group()
    print("[INFO] files per group:", {r: len(v) for r, v in files_by_group.items()})

    draw_hist_grid(METRICS_SET1, OUT_PNG_SET1, files_by_group)
    draw_hist_grid(METRICS_SET2, OUT_PNG_SET2, files_by_group)



In [ ]:
main()
